In [1]:
!git clone https://github.com/unknowne5/TinyRecursiveModels.git
%cd TinyRecursiveModels

Cloning into 'TinyRecursiveModels'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (366/366), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 366 (delta 241), reused 304 (delta 179), pack-reused 0 (from 0)
Receiving objects: 100% (366/366), 3.31 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (241/241), done.
/content/TinyRecursiveModels


In [ ]:
!pip install argdantic

In [12]:
!rm -rf data/vision_qa
!python3 dataset/build_vision_qa_dataset.py

Generating Vision Q&A dataset...
  Generating train example 0/8000
  Generating train example 1000/8000
  Generating train example 2000/8000
  Generating train example 3000/8000
  Generating train example 4000/8000
  Generating train example 5000/8000
  Generating train example 6000/8000
  Generating train example 7000/8000

Generated train split: 8000 examples
  Generating test example 0/1500
  Generating test example 1000/1500

Generated test split: 1500 examples

Dataset generated successfully!


In [3]:
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [9]:
!git fetch --all
!git diff main origin/main
!git merge origin/main
!git show --summary

Fetching origin
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 419 bytes | 419.00 KiB/s, done.
From https://github.com/unknowne5/TinyRecursiveModels
   4a3b108..f3fe1b6  main       -> origin/main
diff --git a/dataset/build_vision_qa_dataset.py b/dataset/build_vision_qa_dataset.py
index 0278f60..f72f8b0 100644
--- a/dataset/build_vision_qa_dataset.py
+++ b/dataset/build_vision_qa_dataset.py
@@ -128,6 +128,10 @@ def generate_dataset(config: VisionDatasetConfig, split: str, token_to_id: Dict[
     puzzle_identifiers, puzzle_indices, group_indices = [], [0], [0]
     debug_samples = []
     
+    seq_len_patches = (config.image_size // config.patch_size) ** 2
+    # Total seq len is text tokens + image patches
+    total_seq_len = 16 + seq_len_patches
+    
     for i in range(num_puzzles):
         if i % 100

In [13]:
!DISABLE_COMPILE=1 uv run python pretrain.py \
    arch=trm \
    data_paths="[data/vision_qa]" \
    +arch.is_vision=True \
    +arch.patch_size=4 \
    arch.halt_exploration_prob=0.1 \
    arch.halt_max_steps=12 \
    arch.H_cycles=2 \
    arch.L_cycles=2 \
    arch.H_layers=0 \
    arch.L_layers=2 \
    arch.hidden_size=64 \
    arch.num_heads=2 \
    arch.expansion=2 \
    arch.puzzle_emb_ndim=4 \
    arch.forward_dtype=float32 \
    arch.puzzle_emb_len=4 \
    global_batch_size=512 \
    epochs=1500 \
    lr=0.001 \
    puzzle_emb_lr=0.01 \
    weight_decay=0.001 \
    puzzle_emb_weight_decay=0.0 \
    lr_warmup_steps=0 \
    eval_interval=1 \
    +run_name="vision_qa_r1" \
    +entity="gunalevishu-test" \
    +project_name="trm"

Failed to import AdamATan2. Using AdamW instead
Using device: cuda
📁 Checkpoint directory: checkpoints/trm/vision_qa_r1
No evaluator found
creating model with config.global_batch_size=512 train_metadata.vocab_size=32 and train_metadata.num_puzzle_identifiers=1
TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (patch_embed): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4))
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
     

In [14]:
import os
import glob
from google.colab import files
import zipfile

def get_latest_checkpoint_path(directory_path: str) -> str | None:
    """
    Finds the path of the latest checkpoint in a given directory.

    The latest checkpoint is determined by the highest step number in the filename,
    assuming filenames are in the format 'step_<number>' or 'final_step_<number>'.

    Args:
        directory_path: The path to the directory containing the checkpoints.

    Returns:
        The full path to the latest checkpoint file, or None if no checkpoints are found.
    """
    if not os.path.isdir(directory_path):
        print(f"Error: Directory not found at '{directory_path}'")
        return None

    # Case 1: Look for files like 'step_1000', 'step_2000'
    step_files = [f for f in os.listdir(directory_path) if f.startswith('step_') and os.path.isfile(os.path.join(directory_path, f))]

    latest_step_file = None
    max_step = -1

    if step_files:
        try:
            latest_step_file = max(step_files, key=lambda x: int(x.split('_')[1]))
            max_step = int(latest_step_file.split('_')[1])
        except (ValueError, IndexError):
            print(f"Could not parse step number from 'step_' checkpoint filenames in '{directory_path}'")

    # Case 2: Look for final checkpoint directories like 'final_step_300'
    final_step_dirs = glob.glob(os.path.join(directory_path, 'final_step_*'))

    latest_final_dir = None
    max_final_step = -1

    if final_step_dirs:
        try:
            latest_final_dir = max(final_step_dirs, key=lambda x: int(x.split('_')[-1]))
            max_final_step = int(latest_final_dir.split('_')[-1])
        except (ValueError, IndexError):
            print(f"Could not parse step number from 'final_step_' checkpoint directories in '{directory_path}'")

    # Compare and return the latest
    if max_step == -1 and max_final_step == -1:
        print(f"No valid checkpoint files or directories found in '{directory_path}'")
        return None

    if max_final_step > max_step:
        model_path = os.path.join(latest_final_dir, 'model.pt')
        if os.path.exists(model_path):
            return model_path
        else:
            print(f"Found final checkpoint directory '{latest_final_dir}' but 'model.pt' is missing.")
            if latest_step_file:
                return os.path.join(directory_path, latest_step_file)
            return None
    else:
        return os.path.join(directory_path, latest_step_file)

# --- Main Script ---

checkpoint_dir = '/content/TinyRecursiveModels/checkpoints/trm/vision_qa_r1'
latest_checkpoint_path = get_latest_checkpoint_path(checkpoint_dir)

if latest_checkpoint_path:
    print(f"Latest checkpoint found at: {latest_checkpoint_path}")

    # --- 1. Identify all checkpoint artifacts ---

    all_step_files = glob.glob(os.path.join(checkpoint_dir, 'step_*'))
    all_final_step_dirs = glob.glob(os.path.join(checkpoint_dir, 'final_step_*'))
    all_checkpoint_artifacts = set(all_step_files + all_final_step_dirs)

    # --- 2. Identify the *container* of the latest checkpoint ---

    latest_artifact_path = None
    if 'final_step_' in latest_checkpoint_path and os.path.basename(latest_checkpoint_path) == 'model.pt':
        latest_artifact_path = os.path.dirname(latest_checkpoint_path)
    elif 'step_' in os.path.basename(latest_checkpoint_path):
        latest_artifact_path = latest_checkpoint_path

    if not latest_artifact_path:
        print("Error: Could not determine the container for the latest checkpoint. Aborting zip.")
    else:
        # --- 3. Determine which old artifacts to exclude ---
        old_artifacts = all_checkpoint_artifacts - {latest_artifact_path}
        print(f"Found {len(old_artifacts)} old checkpoints to exclude.")

        # --- 4. Create the selective Zip file ---
        zip_root_folder = os.path.basename(checkpoint_dir)

        # ⭐️⭐️⭐️ CHANGE IS HERE ⭐️⭐️⭐️
        # Extract the step number from the artifact's filename
        # e.g., 'step_4950' -> '4950' or 'final_step_300' -> '300'
        latest_artifact_basename = os.path.basename(latest_artifact_path)
        step_number = latest_artifact_basename.split('_')[-1]

        # Use the extracted step number in the output name
        output_zip_name = f"{zip_root_folder}_step_{step_number}.zip"
        # ⭐️⭐️⭐️ END OF CHANGE ⭐️⭐️⭐️

        print(f"Creating '{output_zip_name}'...")

        with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:

            # Renamed 'files' to 'filenames' to avoid conflict with the 'files' module
            for root, dirs, filenames in os.walk(checkpoint_dir):

                # Prune old checkpoint directories
                dirs[:] = [d for d in dirs if os.path.join(root, d) not in old_artifacts]

                for file in filenames:
                    file_path = os.path.join(root, file)

                    # Skip old 'step_...' checkpoint files
                    if file_path in old_artifacts:
                        continue

                    arcname = os.path.join(zip_root_folder, os.path.relpath(file_path, checkpoint_dir))
                    zipf.write(file_path, arcname)

        print(f"Successfully created '{output_zip_name}'.")

        # --- 5. Download the final Zip file ---
        files.download(output_zip_name)

else:
    print("No latest checkpoint found. Nothing to zip or download.")

Latest checkpoint found at: /content/TinyRecursiveModels/checkpoints/trm/vision_qa_r1/step_3000
Found 199 old checkpoints to exclude.
Creating 'vision_qa_r1_step_3000.zip'...
Successfully created 'vision_qa_r1_step_3000.zip'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>